# Extract

In [2]:
from datetime import datetime, date

print(datetime.now().strftime("%Y%m%d"))

20260430


In [ ]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date
today = str(datetime.now().strftime("%Y%m%d"))
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
import pandas as pd 
import glob
from pyspark.sql.functions import *
from deep_translator import GoogleTranslator
import country_converter as coco
from pyspark.sql.types import StringType
import logging

#region Configuration du logging
#Cette fonction sert à configurer le logger pour les différentes actions de l'ETL.
#Chaque fonction appelera le logger pour enregistrer les différentes étapes de l'ETL et les éventuelles erreurs qui pourraient survenir.
#Si le logger n'a pas été créé, la fonction le créera. Si le logger existe déjà, il sera utilisé tel quel et renvoyé.
def configure_logger(name="etl_daily"):
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True) #Création du dossier de logs s'il n'existe pas
    logger = logging.getLogger(name)

    if not logger.handlers: #Évite de configurer plusieurs fois le logger si la fonction est appelée plusieurs fois
        logger.setLevel(logging.INFO) # DEBUG, INFO, WARNING, ERROR, CRITICAL
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)s | %(name)s | %(message)s" #Format : date | niveau de log (info/warning,error,etc...) | nom du logger (etl_daily) | message
        )

        #définition des handlers
        ##Les handlers permettent de définir où les logs sont envoyés
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)
        file_handler = logging.FileHandler(
            f"{log_dir}/etl_{today}.log",
            mode='a', #mode 'a' pour append, 'w' pour écraser le fichier à chaque exécution
            encoding='utf-8')
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        logger.addHandler(stream_handler)

        logger.propagate = False  #Empêche les logs de remonter à la racine et d'être dupliqués si d'autres loggers sont utilisés dans le projet
    return logger
#endregion

spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


def extract_daily_data(logger=None) :
    if logger is None:
        logger = configure_logger()
    logger.info("Starting data extraction")
    #region Extraction des données
    try : 
        logger.info("Extracting personnel data")
        #Charger les fichiers de personnel
        fichiers_personnel = glob.glob(f'BDD_BGES/**/PERSONNEL_{today}.txt', recursive=True)
        psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
        psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)
    except Exception as e:
        logger.exception("Error during personnel data extraction")
        raise e

    try : 
        logger.info("Extracting informatic furnitures data")
        #Charger les fichiers de matériel informatique
        fichiers_mat_inf = glob.glob(f'BDD_BGES/**/MATERIEL_INFORMATIQUE_{today}.txt', recursive=True)
        psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
        psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

        #Charge les données d'impact carbone du matériel informatique
        psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
        psdf_impact_matinf = psdf_impact_matinf.rename(columns={
            'Type': 'TYPE',
            'Impact': 'IMPACT',
            'Modèle': 'MODELE'
        })

        #Fusionne les données de matériel informatique avec les données d'impact carbone
        psdf_mat_inf = psdf_mat_inf_temp.merge(
            psdf_impact_matinf,
            on=['TYPE','MODELE'],
            how='inner'
        )
        #Impact est en kg/CO2, il faut le convertir en tonnes de CO2
        psdf_mat_inf['IMPACT'] = psdf_mat_inf['IMPACT'] / 1000
        psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')
    except Exception as e:
        logger.exception("Error during informatic furnitures data extraction")
        raise e

    try :
        logger.info("Extracting mission data")
        #Charger les fichiers de mission
        fichiers_mission = glob.glob(f'BDD_BGES/**/MISSION_{today}.txt', recursive=True)
        psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
        psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)
    except Exception as e:
        logger.exception("Error during mission data extraction")
        raise e
    #endregion
    return psdf_personnel, psdf_mat_inf, psdf_mission
psdf_personnel, psdf_mat_inf, psdf_mission = extract_daily_data()

# Transform

In [ ]:
#Mapping des rôles et des types de mission pour les uniformiser dans la base de données
role_mapping = {
    #Economy
    'Ökonom': 'Economist',
    'Economiste': 'Economist',
    'Economist': 'Economist',

    #Engineer
    'Dateningenieur': 'Data Engineer',
    'Ingénieur Data': 'Data Engineer',
    'Data Engineer': 'Data Engineer',

    'Computeringenieur': 'Computer Engineer',
    'Ingénieur Informaticien': 'Computer Engineer',
    'Computer Engineer': 'Computer Engineer',

    #RH
    'Personalleiter': 'HR Manager',
    'DRH': 'HR Manager',
    'HRD': 'HR Manager',

    #Management
    'Führungskraft': 'Manager',
    'Leadership': 'Manager',
    'Cadre': 'Manager',  # choix le plus cohérent globalement

    #Executive
    'Business Executive': 'Business Executive'
}
mapping_list = [[lit(k), lit(v)] for k, v in role_mapping.items()]
mapping_role = create_map(*mapping_list)

type_mission_mapping = {
    #Business
    'Geschäftstreffen': 'Business Meeting',
    'Business Meeting': 'Business Meeting',
    'Rencontre entreprises': 'Business Meeting',

    #Conference
    'Konferenz': 'Conference',
    'Conference': 'Conference',
    'Conférence': 'Conference',

    #Training
    'Schulung': 'Training',
    'Formation': 'Training',
    'Vocational Training': 'Training',

    #Meeting
    'Meeting': 'Internal Meeting',
    'Team Meeting': 'Internal Meeting',
    'Réunion': 'Internal Meeting',

    #Development
    'Entwicklung': 'Development',
    'Development': 'Development',
    'Développement': 'Development'
}
mapping_list = [[lit(k), lit(v)] for k, v in type_mission_mapping.items()]
mapping_type_mission = create_map(*mapping_list)

def transform_daily_data(logger=None):
    if logger is None:
        logger = configure_logger()
    #region Création des tables de faits et de dimensions

    try:
        logger.info("Transforming data and creating fact and dimension tables")
        # Transformation en spark dataframe
        sdf_personnel = psdf_personnel.to_spark(index_col='ID_PERSONNEL')
        sdf_mat_inf = psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO')
        sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')
    except Exception as e:
        logger.exception(f"Error during data transformation: {e}")
        raise e
    
    #Formatage des données dans le format des tables de la base de données
    #Table FAITS
    try : 
        logger.info("Creating table 'FAITS_MISSION'")
        FAITS_MISSION = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select(
            'ID_MISSION',
            'ID_PERSONNEL',
            col('VILLE').alias('SITE'),
            year(to_date(col('DATE_MISSION'))).alias('ANNEE'),
            month(to_date(col('DATE_MISSION'))).alias('MOIS'),
            dayofmonth(to_date(col('DATE_MISSION'))).alias('JOUR')
        )
    except Exception as e:
        logger.exception(f"Error during mission facts table creation: {e}")
        raise e
    
    try : 
        logger.info("Creating table 'FAITS_MATERIEL_INFORMATIQUE'")
        FAITS_MATERIEL_INFORMATIQUE = sdf_mat_inf.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select(
            'ID_PERSONNEL',
            'ID_MATERIELINFO',
            year(to_date(col('DATE_ACHAT'))).alias('ANNEE'),
            month(to_date(col('DATE_ACHAT'))).alias('MOIS'),
            dayofmonth(to_date(col('DATE_ACHAT'))).alias('JOUR'),
            col('VILLE').alias('SITE')
        )
    except Exception as e:
        logger.exception(f"Error during table 'FAITS_MATERIEL_INFORMATIQUE' creation: {e}")
        raise e

    #TABLE SITE
    try :
        logger.info("Creating table 'SITE'")
        SITE = sdf_personnel.select(col('VILLE').alias('SITE')).distinct()
    except Exception as e:
        logger.exception(f"Error during table 'SITE' creation: {e}")
        raise e
    
    #TABLE DATE
    try :
        logger.info("Creating table 'DATE'")
        DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
        DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE'))
        DATE = DATE.select(year(col('DATE')).alias('ANNEE'), month(col('DATE')).alias('MOIS'), dayofmonth(col('DATE')).alias('JOUR'))
    except Exception as e:
        logger.exception(f"Error during table 'DATE' creation: {e}")
        raise e

    #TABLE PERSONNEL
    try : 
        logger.info("Creating table 'PERSONNEL'")
        PERSONNEL = sdf_personnel.select('ID_PERSONNEL','PAYS','FONCTION_PERSONNEL',col('DT_NAISS').alias('DATE_NAISSANCE'))
        #uniformisation des rôles du personnel
        PERSONNEL = PERSONNEL.withColumn(
            "FONCTION_PERSONNEL",
            mapping_role[col("FONCTION_PERSONNEL")]
        )
    except Exception as e:
        logger.exception(f"Error during table 'PERSONNEL' creation: {e}")
        raise e
    
    #TABLE MATERIEL_INFORMATIQUE
    try : 
        logger.info("Creating table 'MATERIEL_INFORMATIQUE'")
        MATERIEL_INFORMATIQUE = sdf_mat_inf.select('ID_MATERIELINFO', 'TYPE', 'MODELE', 'IMPACT')
        MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
            "ECRAN",
            when(lower(col("TYPE")).contains("sans ecran"), "non")
            .otherwise("oui")
        ).withColumn(
            "TYPE_CLEAN",
            trim(split(lower(col("TYPE")), "sans ecran").getItem(0))
        ).select('ID_MATERIELINFO', col('TYPE_CLEAN').alias('TYPE'), 'MODELE', 'IMPACT', 'ECRAN')
    except Exception as e:
        logger.exception(f"Error during table 'MATERIEL_INFORMATIQUE' creation: {e}")
        raise e
    
    #TABLE LOCALISATION
    try : 
        logger.info("Creating table 'LOCALISATION'")
        cc = coco.CountryConverter()

        def get_continent(pays):
            try:
                continent = cc.convert(names=pays, to='continent')
                if continent == "not found":
                    return None
                return continent
            except:
                return None
            
        logger.info("Extracting unique cities and countries from data")
        LOCALISATION = sdf_mission.select(
            col("VILLE_DEPART").alias("VILLE"),
            col("PAYS_DEPART").alias("PAYS")
        ).union(
            sdf_mission.select(
                col("VILLE_DESTINATION").alias("VILLE"),
                col("PAYS_DESTINATION").alias("PAYS")
            )
        ).distinct()

        rows = LOCALISATION.collect()
        pays_uniques = [row["PAYS"] for row in LOCALISATION.select("PAYS").distinct().collect()]
        logger.info("Unique countries found: %s", pays_uniques)
        logger.info("Starting translation of country names to English for continent mapping")
        translator = GoogleTranslator(source='auto', target='en')

        dict_pays_en = {
            pays: ("Sweden" if pays.lower() in ["suede", "suède"] else translator.translate(pays))
            for pays in pays_uniques
        }
        logger.info("Getting continent for each country")
        dict_pays_continent = {
            pays_fr: get_continent(dict_pays_en[pays_fr])
            for pays_fr in dict_pays_en
        }
        mapping_list = []
        for k, v in dict_pays_continent.items():
            mapping_list += [lit(k), lit(v)]

        mapping_expr = create_map(*mapping_list)
        logger.info("Finalization of 'LOCALISATION' table with continent mapping")
        LOCALISATION = LOCALISATION.withColumn(
            "CONTINENT",
            mapping_expr[col("PAYS")]
        )
    except Exception as e:
        logger.exception(f"Error during table 'LOCALISATION' creation: {e}")
        raise e
    #endregion
    return FAITS_MISSION, FAITS_MATERIEL_INFORMATIQUE, SITE, DATE, PERSONNEL, MATERIEL_INFORMATIQUE, LOCALISATION
daily_FAITS_MISSION, daily_FAITS_MATERIEL_INFORMATIQUE, daily_SITE, daily_DATE, daily_PERSONNEL, daily_MATERIEL_INFORMATIQUE, daily_LOCALISATION = transform_daily_data()

